# Case Study 1: Hospital Readmission Prediction

**Course:** Machine Learning Essentials
**Student:** *(write your name and roll number here)*

**Objective:** Use Logistic Regression with L2 regularization on patient records
(diagnosis codes, vitals, prior visits) to predict 30-day hospital readmission risk.
Evaluate the model using ROC-AUC and discuss the clinical cost of
false negatives vs. false positives.

> Note: Since we were not given a specific hospital dataset, this notebook
> generates a small **synthetic** patient dataset that mimics real hospital
> records (diagnosis codes, vitals, prior admissions). The same code will work
> on a real dataset (e.g. the UCI "Diabetes 130-US hospitals" dataset) if you
> just replace the data-loading step.


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else None


## 2. Create the Patient Dataset

Each row = one patient's hospital stay record. Columns:

| Column | Meaning |
|---|---|
| age | patient age (years) |
| prior_admissions | number of hospital admissions in the last 12 months |
| systolic_bp | systolic blood pressure (mm Hg) |
| heart_rate | resting heart rate (bpm) |
| glucose | blood glucose level (mg/dL) |
| diag_diabetes | 1 if primary diagnosis code = diabetes |
| diag_heart_disease | 1 if primary diagnosis code = heart disease |
| diag_respiratory | 1 if primary diagnosis code = respiratory illness |
| length_of_stay | number of days admitted |
| readmitted | **target**: 1 if patient was readmitted within 30 days, else 0 |


In [ ]:
n_patients = 600

age = np.random.randint(20, 90, n_patients)
prior_admissions = np.random.poisson(1.2, n_patients)
systolic_bp = np.random.normal(130, 18, n_patients).round(1)
heart_rate = np.random.normal(80, 12, n_patients).round(1)
glucose = np.random.normal(120, 35, n_patients).round(1)
length_of_stay = np.random.randint(1, 15, n_patients)

# one primary diagnosis code per patient (simplified from real ICD codes)
diagnosis = np.random.choice(
    ['diabetes', 'heart_disease', 'respiratory', 'other'],
    size=n_patients, p=[0.25, 0.25, 0.2, 0.3]
)
diag_diabetes = (diagnosis == 'diabetes').astype(int)
diag_heart_disease = (diagnosis == 'heart_disease').astype(int)
diag_respiratory = (diagnosis == 'respiratory').astype(int)

df = pd.DataFrame({
    'age': age,
    'prior_admissions': prior_admissions,
    'systolic_bp': systolic_bp,
    'heart_rate': heart_rate,
    'glucose': glucose,
    'length_of_stay': length_of_stay,
    'diag_diabetes': diag_diabetes,
    'diag_heart_disease': diag_heart_disease,
    'diag_respiratory': diag_respiratory,
})

# Build the TRUE risk score (this is how we simulate real-world patterns),
# then turn it into a 0/1 readmission label using a logistic function + noise.
risk_score = (
    0.03 * (df['age'] - 50)
    + 0.9  * df['prior_admissions']
    + 0.02 * (df['systolic_bp'] - 130)
    + 0.015* (df['glucose'] - 120)
    + 0.6  * df['diag_heart_disease']
    + 0.4  * df['diag_diabetes']
    - 0.05 * df['length_of_stay']
    - 3.0
)
prob_readmit = 1 / (1 + np.exp(-risk_score))
df['readmitted'] = np.random.binomial(1, prob_readmit)

df.head()


## 3. Basic Exploratory Data Analysis (EDA)

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
print(df['readmitted'].value_counts())
sns.countplot(x='readmitted', data=df)
plt.title('Readmission Class Balance (0 = No, 1 = Yes within 30 days)')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## 4. Preprocessing

- Split data into features (`X`) and target (`y`).
- Split into train/test sets (80/20).
- Scale the numeric features with `StandardScaler` (Logistic Regression is
  sensitive to feature scale, especially when L2 regularization is applied,
  because the penalty is applied equally to all coefficients).


In [ ]:
X = df.drop(columns=['readmitted'])
y = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)


## 5. Logistic Regression with L2 Regularization

`penalty='l2'` is scikit-learn's default. L2 regularization adds a penalty
equal to the sum of squared coefficients to the loss function. This shrinks
large coefficients and helps prevent overfitting, which is important here
because our dataset is small and some features (like prior admissions and
diagnosis) can dominate the model.

`C` is the **inverse** of the regularization strength (small `C` = stronger
regularization).


In [ ]:
model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000)
model.fit(X_train_scaled, y_train)

coeff_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values(by='coefficient', key=abs, ascending=False)

coeff_df


## 6. Model Evaluation

We use **ROC-AUC** as the primary metric (not just accuracy) because:
- The classes may be imbalanced (fewer readmissions than non-readmissions).
- ROC-AUC measures how well the model ranks patients by risk, across all
  possible decision thresholds — which matters a lot in healthcare, where
  the "best" threshold depends on clinical cost, not just accuracy.


In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print()
print('Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print('ROC-AUC Score:', round(auc, 3))


In [ ]:
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - Hospital Readmission Prediction')
plt.legend()
plt.show()


## 7. Clinical Cost of False Negatives vs. False Positives

In the confusion matrix above:

- **False Negative (FN)**: model predicts *"will NOT be readmitted"*, but the
  patient actually *is* readmitted within 30 days.
- **False Positive (FP)**: model predicts *"WILL be readmitted"*, but the
  patient actually is *not*.

**Why False Negatives are usually more dangerous in this case study:**
A false negative means a genuinely high-risk patient is sent home without
extra follow-up care (nurse calls, early check-ups, medication review).
This can lead to missed complications, a real medical emergency, or even
death, and it usually causes a much more expensive/emergency readmission
later. The clinical and financial cost of a missed high-risk patient is
much higher than the cost of a false alarm.

**Why False Positives still matter:**
A false positive means a low-risk patient is incorrectly flagged as
high-risk. This wastes hospital resources (extra follow-up staff time,
phone calls, possibly an unnecessary extended stay) and can also increase
patient anxiety and healthcare costs. So FPs are not free either — they
just tend to be *less costly* than FNs in a clinical setting.

**Practical takeaway:**
Because FN is costlier than FP here, hospitals usually prefer a model
with **high recall (sensitivity)** for the readmitted class, even if that
means accepting more false positives. In practice, this means we would
**lower the classification threshold** below the default 0.5, so that more
borderline patients get flagged as "at risk" and receive extra care ---
trading some false positives for fewer missed high-risk patients.


In [ ]:
# Example: comparing default threshold (0.5) vs a lower threshold (0.3)
# to show how recall for the "readmitted" class improves.
from sklearn.metrics import recall_score, precision_score

for threshold in [0.5, 0.3]:
    y_pred_thresh = (y_prob >= threshold).astype(int)
    recall = recall_score(y_test, y_pred_thresh)
    precision = precision_score(y_test, y_pred_thresh)
    print(f"Threshold = {threshold}: Recall = {recall:.2f}, Precision = {precision:.2f}")


## 8. Conclusion

- We built a Logistic Regression model with L2 regularization to predict
  30-day hospital readmission from patient vitals, prior visit history, and
  diagnosis codes.
- The model achieved an ROC-AUC of the value printed above, which measures
  how well it ranks patients by readmission risk.
- Because missing a high-risk patient (false negative) is clinically more
  costly than a false alarm (false positive), hospitals should tune the
  classification threshold to favor recall over raw accuracy for this
  use case.

**Possible improvements (future work):**
- Use a real hospital dataset (e.g. UCI "Diabetes 130-US hospitals" dataset).
- Try class-weighting (`class_weight='balanced'`) to handle imbalance.
- Compare with other models (Random Forest, XGBoost) and check calibration.
